In [ ]:

import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np

import pickle

from misc_utils import score_colors,score_colors_original

from pydeseq2.preprocessing import deseq2_norm

# Loading

In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
pickle_path = "/home/bmb/haxx/working/ceisel_mumm/gsea_pickles/"
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden_new.h5ad")


In [ ]:
gp.get_library_name(organism="danio rerio")

In [ ]:
zf_wiki = gp.get_library(name='WikiPathways_2018', organism='danio rerio')
zf_kegg = gp.get_library(name='KEGG_2019', organism='danio rerio')
zf_gobp = gp.get_library(name='GO_Biological_Process_2018', organism='danio rerio')

In [ ]:
death_signatures = {
    'parthanatos':['mapk1', 'aif1l', 'dlg4a', 'appa', 'grin2bb', 'appb', 'parp1', 'plat',
       'endog', 'mmp9', 'parga', 'sirt1', 'map2k1', 'dlg4b', 'dlg4b-1', 'il1b',
       'pargl'],
    'necroptosis':['ripk3', 'tradd', 'cylda', 'fadd', 'pgam5', 'igfbp2a', 'cyldl',
       'ripk1l', 'igfbp2b', 'tnfrsf1a', 'vdac2', 'vdac1', 'cyldb', 'rbck1',
       'dnm1l'],
    'apoptosis':['pik3r1', 'prkar1b', 'baxb', 'prkar1aa', 'baxa', 'baxa-1', 'prkacaa',
       'cycsb', 'prkacbb', 'pik3r2', 'ppp3r1b', 'prkacab', 'ikbkb', 'irak1',
       'akt1', 'ppp3cb', 'xiap', 'ppp3r1a', 'prkar1ab', 'bcl2l1', 'birc2',
       'ppp3ca', 'prkacba', 'pik3ca']
}

In [ ]:
pickle.dump(zf_wiki,open(pickle_path + "zf_wiki.pkl","wb"))
pickle.dump(zf_kegg,open(pickle_path + "zf_kegg.pkl","wb"))
pickle.dump(zf_gobp,open(pickle_path + "zf_gobp.pkl","wb"))


# Preprocessing

In [ ]:
zf_combined = zf_wiki | zf_kegg 


In [ ]:
raws = np.array(data.layers["raw"].todense()+1)
normed_raws,size_factors = deseq2_norm(raws)

In [ ]:
normed_counts = pd.DataFrame(normed_raws.T,index=data.var_names,columns=data.obs_names)
normed_counts[~np.isfinite(normed_counts)] = 0
lognorm_counts = np.log2(normed_counts)


# Global

In [ ]:
gs = gsea(
    data=lognorm_counts,
    gene_sets=zf_combined,
    cls = data.obs['exp_condition'],
    # set permutation_type to phenotype if samples >=15
    permutation_type='phenotype',
    permutation_num=1000, # reduce number to speed up test
    outdir=None,
    method='signal_to_noise',
    threads=10, seed= 8
)
gs.pheno_neg="Cntr"
gs.pheno_pos="Mtz"
_res = gs.run()


In [ ]:
pickle.dump(gs,open(pickle_path + "gs_lognorm_combined.pkl","wb"))

In [ ]:
gs = pickle.load(open("./gsea_pickles/gs_lognorm_combined.pkl","rb"))

In [ ]:
terms = gs.res2d.Term
axs = gs.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )

In [ ]:
gs_death_scores = gsea(
    data=lognorm_counts,
    gene_sets=death_signatures,
    cls = data.obs['exp_condition'],
    # set permutation_type to phenotype if samples >=15
    permutation_type='phenotype',
    permutation_num=1000, # reduce number to speed up test
    outdir=None,
    method='signal_to_noise',
    threads=10, seed= 8
)
gs_death_scores.pheno_neg="Cntr"
gs_death_scores.pheno_pos="Mtz"
_res = gs_death_scores.run()

pickle.dump(gs_death_scores,open(pickle_path + "gs_lognorm_death_scores.pkl","wb"))

In [ ]:
# phenotype = [1 if x == "Mtz" else 0 for x in data.obs["exp_condition"]]


from gseapy import gsea
gs = gsea(
    data=normed_counts,
    gene_sets=zf_combined,
    cls = data.obs['exp_condition'],
    # set permutation_type to phenotype if samples >=15
    permutation_type='phenotype',
    permutation_num=1000, # reduce number to speed up test
    outdir=None,
    method='signal_to_noise',
    threads=10, seed= 8
)
gs.pheno_neg="Cntr"
gs.pheno_pos="Mtz"
_res = gs.run()

pickle.dump(gs,open("gs_denorm_combined.pkl","wb"))


In [ ]:
gs = pickle.load(open("gs_denorm_combined.pkl","rb"))

In [ ]:
# terms = gs.res2d.Term
# axs = gs.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )
gs.plot(['parthanatos','necroptosis','apoptosis'])

In [ ]:
phenotype = [1 if x == "Mtz" else 0 for x in data.obs["exp_condition"]]

from gseapy import gsea
gs = gsea(
    data=normed_counts,
    gene_sets=zf_gobp,
    cls = phenotype, # cls=class_vector
    # set permutation_type to phenotype if samples >=15
    permutation_type='phenotype',
    permutation_num=1000, # reduce number to speed up test
    outdir=None,
    method='signal_to_noise',
    threads=10, seed= 8
)
gs_res = gs.run()

pickle.dump(gs,open("gs_denorm_gobp.pkl","wb"))

In [ ]:
gs = pickle.load(open("gs_denorm_gobp.pkl","rb"))

In [ ]:
terms = gs.res2d.Term
axs = gs.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )

# Cell Type Specific

In [ ]:
# General params
gsea_parameters = {
    'permutation_num':1000,
    'method':'signal_to_noise',
    'threads':10,
    'seed':8,
    'outdir':None,
}

def run_pheno(data,phenotype,gene_sets,pickle_path):
    gs = gp.gsea(
        data=data,
        gene_sets=gene_sets,
        cls = phenotype,
        **gsea_parameters
    )
    gs.pheno_neg="Cntr"
    gs.pheno_pos="Mtz"
    gs.run()
    pickle.dump(gs,open(pickle_path,"wb"))
    return gs

def nuke_legend(fig):
    for ax in fig.get_axes():
        l = ax.get_legend()
        if l:
            l.remove()
    return fig


## RGCs

In [ ]:
rgc_mask = data.obs['cell_type'] == "RGCs"
rgc_normed = normed_counts.loc[:,rgc_mask]
rgc_lognormed = lognorm_counts.loc[:,rgc_mask]
rgc_phenotype = np.array(data.obs['exp_condition'])[rgc_mask]


In [ ]:
_ = run_pheno(
    data=rgc_normed,
    phenotype=rgc_phenotype,
    gene_sets=zf_combined,
    pickle_path=pickle_path + "gs_rgc_denorm_combined.pkl"
)

_ = run_pheno(
    data=rgc_lognormed,
    phenotype=rgc_phenotype,
    gene_sets=zf_combined,
    pickle_path=pickle_path + "gs_rgc_lognorm_combined.pkl"
)


In [ ]:
gs = pickle.load(open(pickle_path + "gs_rgc_denorm_combined.pkl","rb"))
terms = gs.res2d.Term
terms = terms[:5]
fig = gs.plot(
    terms, show_ranking=False, legend_kws={'loc': (1.05, 0)})
fig = nuke_legend(fig)
fig.suptitle("RGCs Only, Ablation (pos) vs Control (neg)",fontsize=16,fontweight="bold")
# fig.savefig("./figures/rgc_kegg_wiki.png",dpi=300,bbox_inches="tight")
fig.savefig("./figures/rgc_kegg_wiki_no_legend.png",dpi=300,bbox_inches="tight")
fig.show()

In [ ]:
_ = run_pheno(
    data=rgc_normed,
    phenotype=rgc_phenotype,
    gene_sets=death_signatures,
    pickle_path=pickle_path + "gs_rgc_denorm_death.pkl"
)

_ = run_pheno(
    data=rgc_lognormed,
    phenotype=rgc_phenotype,
    gene_sets=death_signatures,
    pickle_path=pickle_path + "gs_rgc_lognorm_death.pkl"
)


In [ ]:
gs = pickle.load(open(pickle_path + "gs_rgc_denorm_death.pkl","rb"))

terms = gs.res2d.Term
fig = gs.plot(
    terms = ['parthanatos','necroptosis','apoptosis'],
    colors = list(score_colors.values()),
    show_ranking=False,
    legend_kws={'loc': (1.05, 0)}
)
# fig = nuke_legend(fig)
fig.suptitle("RGCs Only, Ablation (pos) vs Control (neg)",fontsize=16,fontweight="bold")
fig.savefig("./figures/rgc_death_signatures.png",dpi=300,bbox_inches="tight")

## Bipolars

In [ ]:
bipolar_subset = data.obs['cell_type'] == "Bipolar cells"
bipolar_normed = normed_counts.loc[:,bipolar_subset]
bipolar_lognormed = lognorm_counts.loc[:,bipolar_subset]
bipolar_phenotype = np.array(data.obs['exp_condition'])[bipolar_subset]

_ = run_pheno(
    data=bipolar_normed,
    phenotype=bipolar_phenotype,
    gene_sets=death_signatures,
    pickle_path=pickle_path + "gs_bipolar_denorm_death.pkl"
)

_ = run_pheno(
    data=bipolar_lognormed,
    phenotype=bipolar_phenotype,
    gene_sets=death_signatures,
    pickle_path=pickle_path + "gs_bipolar_lognorm_death.pkl"
)

_ = run_pheno(
    data=bipolar_normed,
    phenotype=bipolar_phenotype,
    gene_sets=zf_combined,
    pickle_path=pickle_path + "gs_bipolar_denorm_combined.pkl"
)

_ = run_pheno(
    data=bipolar_lognormed,
    phenotype=bipolar_phenotype,
    gene_sets=zf_combined,
    pickle_path=pickle_path + "gs_bipolar_lognorm_combined.pkl"
)


In [ ]:
gs = pickle.load(open(pickle_path + "gs_rgc_denorm_death.pkl","rb"))

terms = gs.res2d.Term
fig = gs.plot(
    terms = ['parthanatos','necroptosis','apoptosis'],
    colors = list(score_colors.values()),
    show_ranking=False,
    legend_kws={'loc': (1.05, 0)}
)
fig.suptitle("RGCs Only, Ablation (pos) vs Control (neg)",fontsize=16,fontweight="bold")
fig.savefig("./figures/rgc_death_signatures.png",dpi=300,bbox_inches="tight")